In [2]:
from pystac_client import Client
from shapely.geometry import box, mapping
import shapely.ops as ops
import pyproj
import planetary_computer
import rasterio
import rasterio.mask
import os
import time
import csv
import pandas as pd
import numpy as np

# --- CONFIG ---
input_csv = "bike_segments.csv"
output_dir = "naip_bike_images"
os.makedirs(output_dir, exist_ok=True)

# --- Load segment metadata with lat/lon/osmid ---
df = pd.read_csv(input_csv)
bike_coords = df[['lat', 'lon', 'segment_index', 'osmid']].values.tolist()

# --- Connect to Planetary Computer STAC API ---
stac = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

# --- Prepare metadata CSV ---
csv_path = os.path.join(output_dir, "image_metadata.csv")
if not os.path.exists(csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "filename", "segment_index", "osmid", "lat", "lon", "year", "item_id",
            "collection", "provider", "bbox", "cloud_cover"
        ])

# --- Geometry projection helper ---
def reproject_geom_to_crs(geom, src_crs, dst_crs):
    transformer = pyproj.Transformer.from_crs(src_crs, dst_crs, always_xy=True)
    return ops.transform(transformer.transform, geom)

def feet_to_degrees(feet):
    """
    Convert feet to approximate degrees (valid for small buffers).
    Assumes 1 degree ≈ 111,000 meters.
    """
    meters = feet * 0.3048
    return meters / 111000

# --- Buffer size: ~400ft ≈ 122m ≈ 0.0011 degrees ---
buffer_deg = feet_to_degrees(250)  # ~0.0011 degrees

# --- Download loop ---
for idx, (lat, lon, segment_index, osmid) in enumerate(bike_coords[:5]):  # ← adjust range
    min_lon, max_lon = lon - buffer_deg, lon + buffer_deg
    min_lat, max_lat = lat - buffer_deg, lat + buffer_deg
    bbox_geom = box(min_lon, min_lat, max_lon, max_lat)

    # 🔍 Query all NAIP images at location
    # Search all images between 2015 and 2024
    search = stac.search(
        collections=["naip"],
        intersects=mapping(bbox_geom),
        datetime="2015-01-01/2024-12-31"
    )

    items = list(search.items())
    if not items:
        print(f"× Node {idx+1}: No NAIP images found.")
        continue

    # Group items by acquisition year
    items_by_year = {}
    for item in items:
        year = item.datetime.year
        if year not in items_by_year:
            items_by_year[year] = item  # store only 1 per year

    sorted_years = sorted(items_by_year.keys(), reverse=True)
    pair_found = False

    # Search for a pair at least 3 years apart
    for i in range(len(sorted_years)):
        for j in range(i + 1, len(sorted_years)):
            y1, y2 = sorted_years[i], sorted_years[j]
            if abs(y2 - y1) >= 3:
                selected_items = [items_by_year[y1], items_by_year[y2]]
                pair_found = True
                break
        if pair_found:
            break

    if not pair_found:
        print(f"× Node {idx+1}: No image pair ≥3 years apart.")
        continue

    # ✨ Take latest and one previous image
    item_pairs = selected_items

    for img_item in item_pairs:
        signed_item = planetary_computer.sign(img_item)
        asset = signed_item.assets["image"]
        url = asset.href
        acq_datetime = signed_item.datetime
        acq_year = acq_datetime.year

        item_id = signed_item.id
        collection = signed_item.collection_id
        provider = signed_item.properties.get("provider", "NAIP")
        bbox = signed_item.bbox
        cloud = signed_item.properties.get("eo:cloud_cover", "N/A")

        filename = f"naip_osmid_{int(osmid)}_{acq_year}.tif"
        out_path = os.path.join(output_dir, filename)

        print(f"→ Segment {segment_index} (OSMID {osmid}) - Year {acq_year}")

        try:
            with rasterio.open(url) as src:
                src_crs = src.crs
                bbox_geom_proj = reproject_geom_to_crs(bbox_geom, "EPSG:4326", src_crs)

                if not box(*src.bounds).intersects(bbox_geom_proj):
                    print(f"⚠️ Skipped: No overlap for {filename}")
                    continue

                out_image, out_transform = rasterio.mask.mask(
                    src, [mapping(bbox_geom_proj)], crop=True
                )
                out_image = np.clip(out_image, 0, 255).astype(np.uint8)

                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": "GTiff",
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
                })

                with rasterio.open(out_path, "w", **out_meta) as dest:
                    dest.write(out_image)

            print(f"✓ Saved: {filename}")

            # 📥 Log metadata
            with open(csv_path, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    filename, segment_index, osmid, lat, lon, acq_datetime.date(), item_id,
                    collection, provider, bbox, cloud
                ])

        except Exception as e:
            print(f"⚠️ Failed to process {filename} → {e}")

        time.sleep(1)  # Be nice to the API

→ Segment 0.0 (OSMID 49379060.0) - Year 2023
✓ Saved: naip_osmid_49379060_2023.tif
→ Segment 0.0 (OSMID 49379060.0) - Year 2018
✓ Saved: naip_osmid_49379060_2018.tif
→ Segment 1.0 (OSMID 12637095735.0) - Year 2023
✓ Saved: naip_osmid_12637095735_2023.tif
→ Segment 1.0 (OSMID 12637095735.0) - Year 2018
✓ Saved: naip_osmid_12637095735_2018.tif
→ Segment 2.0 (OSMID 898675366.0) - Year 2023
✓ Saved: naip_osmid_898675366_2023.tif
→ Segment 2.0 (OSMID 898675366.0) - Year 2018
✓ Saved: naip_osmid_898675366_2018.tif
→ Segment 3.0 (OSMID 9690954976.0) - Year 2023
✓ Saved: naip_osmid_9690954976_2023.tif
→ Segment 3.0 (OSMID 9690954976.0) - Year 2018
✓ Saved: naip_osmid_9690954976_2018.tif
→ Segment 4.0 (OSMID 5928355089.0) - Year 2023
✓ Saved: naip_osmid_5928355089_2023.tif
→ Segment 4.0 (OSMID 5928355089.0) - Year 2018
✓ Saved: naip_osmid_5928355089_2018.tif
